# Multibrot deep-dive movie studio

Generates long, high-resolution zoom movies into the Multibrot sets, `z → z^n + c`,
on a Colab GPU. The defaults produce a five-minute 1080p dive; the same pipeline
does 4K, 60 fps, and magnifications far past anything float64 can address.

### What it does

| | |
|---|---|
| **Depth** | A perturbation engine with rebasing renders against a reference orbit carried in arbitrary precision. Plain float64 gives out at about 10⁹×; this reaches 10³⁰⁰×. Four coordinates are shipped, verified to the depth they are labelled with, and section 7 searches for more. |
| **Speed** | Whole exponents iterate by repeated complex multiplication instead of `pow`/`sin`/`cos`, interior pixels exit early by cycle detection, colours are mapped on the GPU, and frames go straight into `ffmpeg` — no `FuncAnimation`, no base64 in the notebook. |
| **Looks** | Smooth (fractional) escape counts, a distance estimator that keeps filaments crisp at any depth, cyclic palettes blended in linear light, and supersampled anti-aliasing. |
| **Not guessing** | Section 10 measures the iteration budget this dive actually needs and times real frames before you commit. Section 8 checks the kernels against each other. |
| **Survivability** | A five-minute render outlives most Colab sessions. Frames are encoded into short segments, so re-running picks up where it stopped. |

### Running it

Run sections 1–5 in order, look at the preview in section 6, check section 8, then let
section 10 tell you how long the real thing will take before you start it in section 11.
Everything reads one `CFG` object, so changing a setting in section 2 and re-running
from there is always enough.

**Use an A100.** Everything here is float64, and T4 / L4 / RTX cards run float64 at
1/32 of their float32 rate, which turns a one-hour render into a day. *Runtime →
Change runtime type → A100 GPU*.

**A five-minute deep dive is an hours-long render.** That is the honest cost of a few
million pixels a frame at tens of thousands of iterations each. Section 10 gives you
the number up front; `SUPERSAMPLE` is the strongest lever on it, then `QUALITY`.

## 1 · Setup

Probes the GPU, installs `ffmpeg` and `mpmath` if they are missing, and warns about
float64-limited hardware.

In [ ]:
#@title 1 · Environment setup — run me first
"""Probe the runtime, install what is missing, and report what this machine can do."""

import os, sys, math, json, time, shutil, subprocess, hashlib, textwrap
from dataclasses import dataclass, field, asdict, replace
from pathlib import Path

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import mpmath
except ImportError:
    _pip("mpmath"); import mpmath

try:                       # optional, but makes the reference orbit several times faster
    import gmpy2; HAVE_GMPY2 = True
except ImportError:
    HAVE_GMPY2 = False

try:
    from tqdm.auto import tqdm
except ImportError:
    _pip("tqdm"); from tqdm.auto import tqdm

import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- GPU ----
try:
    import cupy as cp
    _props   = cp.cuda.runtime.getDeviceProperties(0)
    GPU_NAME = _props["name"].decode()
    GPU_CC   = float(f"{_props['major']}.{_props['minor']}")
    GPU_MEM  = cp.cuda.Device(0).mem_info[1] / 2**30
    HAVE_GPU = True
except Exception as _e:                                    # noqa: BLE001
    cp, HAVE_GPU = None, False
    GPU_NAME, GPU_CC, GPU_MEM = "(none)", 0.0, 0.0
    print("!! No usable CuPy/CUDA device:", _e)

# Everything here is float64.  Consumer and inference cards run fp64 at 1/32 or
# 1/64 of fp32, which turns a five minute movie into an overnight job.

FP64_OK = HAVE_GPU and not any(t in GPU_NAME.upper() for t in ("T4", "L4", "RTX", "GTX", "A10"))

# ------------------------------------------------------------- ffmpeg ----
if shutil.which("ffmpeg") is None:
    print("Installing ffmpeg ...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg",
                   shell=True, check=False)
FFMPEG = shutil.which("ffmpeg")
IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")

print("=" * 66)
print(f"  GPU        : {GPU_NAME}   (compute {GPU_CC}, {GPU_MEM:.1f} GB)")
print(f"  CuPy       : {cp.__version__ if HAVE_GPU else 'unavailable'}")
print(f"  ffmpeg     : {FFMPEG or 'MISSING - encoding will fail'}")
print(f"  mpmath     : {mpmath.__version__}" + ("  (+gmpy2)" if HAVE_GMPY2 else "  (no gmpy2)"))
print(f"  Colab      : {IN_COLAB}")
print("=" * 66)
if HAVE_GPU and not FP64_OK:
    print(textwrap.dedent(f"""\
        NOTE  {GPU_NAME} has heavily reduced float64 throughput (typically 1/32 of
              float32).  Deep-dive rendering is all float64, so expect this card
              to be roughly 10-30x slower than an A100.  Either pick a shorter
              DURATION / smaller WIDTH below, or switch the Colab runtime to an
              A100 (Runtime > Change runtime type).
    """))
if not HAVE_GMPY2:
    print("TIP   `!pip install gmpy2` speeds up the high-precision reference orbit.")

## 2 · Configuration

Everything about the movie lives here. The rest of the notebook reads `CFG` and
nothing else, so changing a value and re-running from this point is enough.

Two knobs set the character of the dive: `FINAL_ZOOM`, the magnification of the last
frame as a power of ten, and `DURATION_SECONDS`. Together they fix the pace — 40
decades over 300 seconds is about one doubling every 2.3 seconds, which is a
comfortable cinematic speed. Section 5 prints the rate it works out to.

`FINAL_ZOOM` beyond a target's verified depth is not automatically interesting: the
dive may bottom out inside a solid region. Section 7 both warns about this and
searches for a coordinate that stays alive as deep as you asked for.

In [ ]:
#@title 2 · Movie configuration { display-mode: "form", run: "auto" }

#@markdown ### The fractal
#@markdown Exponent `n` in `z -> z^n + c`.  Whole numbers >= 2 unlock the deep-zoom
#@markdown engine and the distance-estimator shading; fractional or negative values
#@markdown still render, but only to a view width of about `1e-9`.
EXPONENT = 8  #@param {type:"number"}

#@markdown ### Where to dive
#@markdown `curated` uses a coordinate shipped with this notebook, `auto` searches for a
#@markdown fresh one in section 6, `custom` uses the strings below (give them as many
#@markdown digits as you like — they are parsed at full precision).
TARGET = "curated"  #@param ["curated", "auto", "custom"]
CURATED = "z8 · verified to 1e14x"  #@param ["z8 · verified to 1e14x", "z3 · verified to 1e16x", "z2 · verified to 1e16x"]
CENTER_RE = "0.40354650430320575704712382503203116357326819809046312170119304195642699659008171"  #@param {type:"string"}
CENTER_IM = "0.63595924929010483506175432921736501157283176844709732258003703620222994779572375"  #@param {type:"string"}

#@markdown ### Shot
#@markdown `FINAL_ZOOM` is the magnification at the end, as a power of ten: 30 means the
#@markdown last frame is 1e30 times narrower than the first. The shipped `z^8` coordinate
#@markdown is good to 1e44, but a first render is worth keeping finishable.
START_SPAN = 3.0  #@param {type:"number"}
FINAL_ZOOM = 14  #@param {type:"slider", min:2, max:200, step:1}
DURATION_SECONDS = 300  #@param {type:"number"}
FPS = 30  #@param [24, 30, 60] {type:"raw"}
HOLD_END_SECONDS = 3.0  #@param {type:"number"}
EASE_IN = 0.15  #@param {type:"slider", min:0, max:0.5, step:0.01}
EASE_OUT = 0.10  #@param {type:"slider", min:0, max:0.5, step:0.01}
ROTATION_TURNS = 0.25  #@param {type:"number"}

#@markdown ### Picture
QUALITY = "1080p"  #@param ["720p", "1080p", "1440p", "4K"]
SUPERSAMPLE = 2  #@param [1, 2, 3] {type:"raw"}

#@markdown ### Colour
PALETTE = "ember"  #@param ["ember", "aurora", "twilight", "goldleaf", "abyss", "neon", "copper", "pastel"]
COLOR_MODE = "log"  #@param ["log", "sqrt", "linear"]
COLOR_DENSITY = 4.0  #@param {type:"number"}
COLOR_CYCLES = 3.0  #@param {type:"number"}
COLOR_PHASE = 0.0  #@param {type:"number"}
DE_SHADING = True  #@param {type:"boolean"}
EXPOSURE = 1.0  #@param {type:"number"}
INTERIOR_COLOR = "#05040a"  #@param {type:"string"}

#@markdown ### Encoding
CRF = 16  #@param {type:"slider", min:10, max:28, step:1}
X264_PRESET = "slow"  #@param ["ultrafast", "veryfast", "medium", "slow", "slower"]
OUTPUT_DIR = "/content/multibrot_render"  #@param {type:"string"}

# =========================================================================

RESOLUTIONS = {"720p": (1280, 720), "1080p": (1920, 1080),
               "1440p": (2560, 1440), "4K": (3840, 2160)}

# Deep-dive coordinates found with the descent search in section 6.  `zoom` is the
# log10 magnification at which the point was still resolving new structure.
TARGETS = {
    "z8 · verified to 1e14x": {
        "exponent": 8,
        "re": "0.40354650430320575704712382503203116357326819809046312170119304195642699659008171",
        "im": "0.63595924929010483506175432921736501157283176844709732258003703620222994779572375",
        "zoom": 14
    },
    "z3 · verified to 1e16x": {
        "exponent": 3,
        "re": "-0.13476661867959238679318123566019949066685336883054534020727873327361072112920492",
        "im": "0.81899615359403709043475960704938643175412158727512116541914545035703403089989356",
        "zoom": 16
    },
    "z2 · verified to 1e16x": {
        "exponent": 2,
        "re": "-0.64341630722993847909738085455444434046511106466168785189047155395861375569587491",
        "im": "-0.36135443805453469753516393403458550892538033702952977191166000359637315549647372",
        "zoom": 16
    }
}


@dataclass
class MovieConfig:
    exponent: float = 8.0
    center_re: str = "0"
    center_im: str = "0"
    start_span: float = 3.0
    final_zoom: float = 40.0          # log10 magnification at the last frame
    duration: float = 300.0
    fps: int = 30
    hold_end: float = 3.0
    ease_in: float = 0.15
    ease_out: float = 0.10
    rotation_turns: float = 0.0
    width: int = 1920
    height: int = 1080
    supersample: int = 2
    # colour
    palette: str = "ember"
    color_mode: str = "log"
    color_density: float = 4.0
    color_cycles: float = 3.0
    color_phase: float = 0.0
    color_offset: float = 1.5
    de_shading: bool = True
    de_softness: float = 1.0
    de_floor: float = 0.10
    exposure: float = 1.0
    gamma: float = 2.2
    interior_color: str = "#05040a"
    # Iteration schedule: max_iter = base + scale * depth**power, where depth is the
    # decades of magnification so far.  Deep zooms are hungrier than they look — at
    # 1e44x the boundary here needs around 60k iterations, and starving it paints fake
    # interior blobs that shimmer between frames.  Section 10 measures what this
    # particular dive needs and rewrites these.
    base_iter: int = 800
    iter_scale: float = 1200.0
    iter_power: float = 1.0
    iter_cap: int = 400000
    # encoding
    crf: int = 16
    x264_preset: str = "slow"
    x264_tune: str = "animation"
    segment_seconds: int = 10
    outdir: str = "/content/multibrot_render"

    # ---------------------------------------------------------------- derived
    @property
    def center(self):
        mpmath.mp.dps = max(50, self.precision)
        return mpmath.mpc(mpmath.mpf(self.center_re), mpmath.mpf(self.center_im))

    @property
    def final_span(self):
        return self.start_span / 10.0 ** self.final_zoom

    @property
    def precision(self):
        return int(self.final_zoom) + 30

    @property
    def total_frames(self):
        return int(round(self.duration * self.fps)) + int(round(self.hold_end * self.fps))

    @property
    def integer_exponent(self):
        p = round(self.exponent)
        return int(p) if abs(self.exponent - p) < 1e-12 and p >= 2 else 0

    def fingerprint(self):
        """Identifies a render, so a resumed run cannot splice mismatched frames."""
        d = {k: v for k, v in asdict(self).items() if k not in ("outdir", "segment_seconds")}
        return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:16]

    def validate(self):
        problems, notes = [], []
        if self.width % 2 or self.height % 2:
            problems.append("WIDTH and HEIGHT must both be even for yuv420p output.")
        if self.duration <= 0 or self.fps <= 0:
            problems.append("DURATION_SECONDS and FPS must be positive.")
        if not self.integer_exponent:
            if self.final_zoom > 9:
                notes.append(f"Exponent {self.exponent} is not a whole number >= 2, so the "
                             f"perturbation engine is unavailable; FINAL_ZOOM capped at 9 "
                             f"(was {self.final_zoom:g}).")
                self.final_zoom = 9.0
            if self.de_shading:
                notes.append("Distance-estimator shading needs a whole exponent; disabled.")
                self.de_shading = False
        if self.final_span < 1e-300:
            problems.append(f"FINAL_ZOOM {self.final_zoom:g} drives the view width below "
                            f"1e-300, past what float64 deltas can represent. Use <= "
                            f"{math.log10(self.start_span) + 295:.0f}.")
        return problems, notes


_w, _h = RESOLUTIONS[QUALITY]
_verified = None
if TARGET == "curated":
    _t = TARGETS[CURATED]
    _cre, _cim, _exp, _verified = _t["re"], _t["im"], _t["exponent"], _t["zoom"]
    if abs(float(EXPONENT) - _exp) > 1e-9:
        print(f"note  · TARGET is \"curated\", so the exponent comes from {CURATED!r} "
              f"(z^{_exp:g}) and EXPONENT={EXPONENT:g} is ignored.\n"
              f"        Set TARGET to \"auto\" or \"custom\" to choose your own exponent.")
else:                                   # "custom", and "auto" until section 7 fills it in
    _cre, _cim, _exp = CENTER_RE, CENTER_IM, EXPONENT

CFG = MovieConfig(
    exponent=float(_exp),
    center_re=str(_cre), center_im=str(_cim),
    start_span=float(START_SPAN), final_zoom=float(FINAL_ZOOM),
    duration=float(DURATION_SECONDS), fps=int(FPS), hold_end=float(HOLD_END_SECONDS),
    ease_in=float(EASE_IN), ease_out=float(EASE_OUT), rotation_turns=float(ROTATION_TURNS),
    width=_w, height=_h, supersample=int(SUPERSAMPLE),
    palette=PALETTE, color_mode=COLOR_MODE, color_density=float(COLOR_DENSITY),
    color_cycles=float(COLOR_CYCLES), color_phase=float(COLOR_PHASE),
    de_shading=bool(DE_SHADING), exposure=float(EXPOSURE), interior_color=INTERIOR_COLOR,
    crf=int(CRF), x264_preset=X264_PRESET, outdir=OUTPUT_DIR,
)

_problems, _notes = CFG.validate()
if _verified is not None and CFG.final_zoom > _verified:
    _notes.append(f"FINAL_ZOOM {CFG.final_zoom:g} goes past the {_verified:g} decades this "
                  f"coordinate was verified to. The dive may bottom out in a featureless "
                  f"region — section 7 checks, and can search for a deeper target.")
for _n in _notes:
    print("note  ·", textwrap.fill(_n, 74, subsequent_indent="        "))
if _problems:
    for _p in _problems:
        print("ERROR ·", textwrap.fill(_p, 74, subsequent_indent="        "))
    raise ValueError("Fix the configuration above before continuing.")

print(f"""
  z -> z^{CFG.exponent:g} + c
  centre     {CFG.center_re[:34]}
             {CFG.center_im[:34]}
  view width {CFG.start_span:g}  ->  {CFG.final_span:.3e}   ({CFG.final_zoom:g} decades)
  frames     {CFG.total_frames}  ({CFG.duration:g}s + {CFG.hold_end:g}s hold @ {CFG.fps}fps)
  picture    {CFG.width}x{CFG.height}, {CFG.supersample}x{CFG.supersample} supersampled
             ({CFG.width*CFG.supersample*CFG.height*CFG.supersample/1e6:.1f} Mpx computed per frame)
  engine     {'perturbation (arbitrary depth)' if CFG.integer_exponent else 'direct float64 (shallow only)'}
  output     {CFG.outdir}
  id         {CFG.fingerprint()}
""")

## 3 · The engine

Two CUDA kernels behind one `render()` call.

`mb_direct` is the straightforward iteration in float64 and handles any real
exponent, integer or not. It is what the original notebook did, plus a smooth escape
count and a distance estimate, and coordinates computed on the device instead of a
`meshgrid` (a 4K frame at 2× supersampling would otherwise need half a gigabyte just
for `C`).

`mb_perturb` is what makes a real deep dive possible. Below a view width of about
`1e-9` the spacing between neighbouring pixels drops under the rounding error of the
coordinates themselves, and the direct kernel returns mush. Instead one reference
orbit is iterated at hundreds of decimal digits on the CPU, and each pixel tracks
only its *difference* from that orbit — which stays comfortably inside float64.
Because `(Z+d)^p − Z^p` expands binomially and exactly for whole `p`, this is not an
approximation. Rebasing (restarting the reference whenever the true orbit falls below
the delta) is what keeps it free of the glitches that plague naive perturbation.

The whole movie shares one centre, so the reference orbit is computed once and reused
for every frame.

Both kernels stop early on pixels whose orbit has settled onto a cycle, detected with
Brent's algorithm. Deep views are mostly interior, and an interior pixel would
otherwise spend the entire iteration budget proving something that was decided in the
first few hundred steps. Section 8 checks that the optimisation leaves every pixel bit
for bit unchanged.

In [ ]:
#@title 3 · GPU engine — escape-time kernels

CUDA_SOURCE = r"""// =====================================================================
//  Multibrot escape-time kernels:  z <- z^n + c
//
//  mb_direct   double-precision iteration.  Any real exponent n
//              (integer n uses binary exponentiation; otherwise
//              De Moivre).  Good to a view width of ~1e-9.
//  mb_perturb  perturbation + rebasing against a high-precision
//              reference orbit.  Integer n >= 2 only.  Accurate to
//              view widths of ~1e-300, which is what a real deep
//              dive needs.
//
//  Both kernels emit a smooth (fractional) escape count and the log
//  of the distance estimate measured in pixels.
// =====================================================================

#define RESCALE_LIMIT   1e150
#define RESCALE_SHIFT   512
#define LN2             0.69314718055994530942

// z^k for integer k >= 0, by binary exponentiation.
__device__ __forceinline__
void cpowi(double zr, double zi, int k, double *orr, double *oi) {
    double wr = 1.0, wi = 0.0, br = zr, bi = zi;
    while (k) {
        if (k & 1) { double t = wr * br - wi * bi; wi = wr * bi + wi * br; wr = t; }
        double t = br * br - bi * bi; bi = 2.0 * br * bi; br = t;
        k >>= 1;
    }
    *orr = wr; *oi = wi;
}

// Shared tail: turn the final state into (smooth iteration, log pixel-distance).
__device__ __forceinline__
void emit(int escaped, int it, double m2, double dzr, double dzi, int dzexp,
          double log_bail, double log_n, double step, int want_de,
          float *out_nu, float *out_de, int idx) {
    if (!escaped) { out_nu[idx] = -1.0f; out_de[idx] = 0.0f; return; }
    double lz = 0.5 * log(m2);                       // log|z|
    double nu = (double)(it + 1);
    if (log_n > 1e-9) nu -= log(lz / log_bail) / log_n;
    out_nu[idx] = (float)nu;
    if (want_de) {
        double h = hypot(dzr, dzi);
        double ld = (h > 0.0 ? log(h) : -700.0) + (double)dzexp * LN2;
        // distance ~ |z| log|z| / |dz|;  report it in pixels, as a log.
        out_de[idx] = (float)(lz + log(lz) - ld - log(step));
    } else {
        out_de[idx] = 0.0f;
    }
}

extern "C" __global__
void mb_direct(double cx, double cy, double step, double ca, double sa,
               double n, int ip,
               int W, int H, int max_iter,
               double bail2, double log_bail, double log_n,
               int want_de, int check_period,
               float *out_nu, float *out_de)
{
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    if (idx >= W * H) return;
    int py = idx / W;
    int px = idx - py * W;

    double u = ((double)px - 0.5 * (double)(W - 1)) * step;
    double v = ((double)py - 0.5 * (double)(H - 1)) * step;
    double cre = cx + (ca * u - sa * v);
    double cim = cy - (sa * u + ca * v);          // row 0 is the top of the frame

    // A negative exponent makes z = 0 a pole, so start the orbit at c instead.
    double zr = (n < 0.0) ? cre : 0.0;
    double zi = (n < 0.0) ? cim : 0.0;
    double dzr = 0.0, dzi = 0.0;
    int dzexp = 0;

    // Brent cycle detection: an interior point settles onto an attracting cycle, so
    // once z returns to a value it already took, it will never escape and the
    // remaining iterations are wasted.  Deep views are mostly interior, so this is
    // usually the difference between a render that finishes and one that does not.
    double sr = 0.0, si = 0.0;                    // last checkpointed z
    int since = 0, next_check = 1;

    double m2 = 0.0;
    int escaped = 0, it = 0;

    for (it = 0; it < max_iter; ++it) {
        double pr, pi;                            // z^(n-1)
        if (ip > 0) {
            cpowi(zr, zi, ip - 1, &pr, &pi);
        } else {
            double r  = hypot(zr, zi);
            double th = atan2(zi, zr);
            double rm = pow(r, n - 1.0);
            pr = rm * cos((n - 1.0) * th);
            pi = rm * sin((n - 1.0) * th);
        }
        if (want_de) {                            // dz <- n z^(n-1) dz + 1
            double tr = pr * dzr - pi * dzi;
            double ti = pr * dzi + pi * dzr;
            dzr = n * tr + ldexp(1.0, -dzexp);
            dzi = n * ti;
            double a = fmax(fabs(dzr), fabs(dzi));
            if (a > RESCALE_LIMIT) {              // keep |dz| inside double's range
                dzr = ldexp(dzr, -RESCALE_SHIFT);
                dzi = ldexp(dzi, -RESCALE_SHIFT);
                dzexp += RESCALE_SHIFT;
            }
        }
        double nzr = pr * zr - pi * zi + cre;     // z <- z^(n-1) * z + c
        double nzi = pr * zi + pi * zr + cim;
        zr = nzr; zi = nzi;

        m2 = zr * zr + zi * zi;
        if (!(m2 <= bail2)) {                     // also catches inf and nan
            escaped = 1;
            if (!(m2 < 1.0e308)) m2 = bail2 * 4.0; // pin inf/nan to a usable magnitude
            break;
        }
        if (check_period) {
            double ar = zr - sr, ai = zi - si;
            if (ar * ar + ai * ai < 1.0e-32) { it = max_iter; break; }   // periodic
            if (++since >= next_check) { since = 0; next_check <<= 1; sr = zr; si = zi; }
        }
    }
    emit(escaped, it, m2, dzr, dzi, dzexp, log_bail, log_n, step, want_de,
         out_nu, out_de, idx);
}

extern "C" __global__
void mb_perturb(const double *__restrict__ orbit, int olen,
                const double *__restrict__ binom, int p,
                double dcx0, double dcy0,
                double step, double ca, double sa,
                int W, int H, int max_iter,
                double bail2, double log_bail, double log_n,
                int want_de, int check_period,
                float *out_nu, float *out_de)
{
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    if (idx >= W * H) return;
    int py = idx / W;
    int px = idx - py * W;

    double u = ((double)px - 0.5 * (double)(W - 1)) * step;
    double v = ((double)py - 0.5 * (double)(H - 1)) * step;
    double dcr = dcx0 + (ca * u - sa * v);        // offset from the reference point
    double dci = dcy0 - (sa * u + ca * v);

    double dr = 0.0, di = 0.0;                    // d = z - Z[m]
    int m = 0;
    double dzr = 0.0, dzi = 0.0;
    int dzexp = 0;

    double sr = 0.0, si = 0.0;                    // Brent checkpoint, as above
    int since = 0, next_check = 1;

    double m2 = 0.0;
    int escaped = 0, it = 0;

    for (it = 0; it < max_iter; ++it) {
        double Zr = orbit[2 * m], Zi = orbit[2 * m + 1];

        if (want_de) {
            double fr = Zr + dr, fi = Zi + di;    // the true orbit value at this step
            double pr, pi;
            cpowi(fr, fi, p - 1, &pr, &pi);
            double tr = pr * dzr - pi * dzi;
            double ti = pr * dzi + pi * dzr;
            dzr = (double)p * tr + ldexp(1.0, -dzexp);
            dzi = (double)p * ti;
            double a = fmax(fabs(dzr), fabs(dzi));
            if (a > RESCALE_LIMIT) {
                dzr = ldexp(dzr, -RESCALE_SHIFT);
                dzi = ldexp(dzi, -RESCALE_SHIFT);
                dzexp += RESCALE_SHIFT;
            }
        }

        // d <- (Z+d)^p - Z^p + dc, expanded binomially and evaluated by Horner:
        //   sum_{k=1..p} C(p,k) Z^(p-k) d^k  =  d * P(d)
        double Pr = 0.0, Pi = 0.0, wr = 1.0, wi = 0.0;
        for (int k = p; k >= 1; --k) {
            double ar = binom[k] * wr, ai = binom[k] * wi;
            double tr = dr * Pr - di * Pi;
            double ti = dr * Pi + di * Pr;
            Pr = ar + tr; Pi = ai + ti;
            double nwr = wr * Zr - wi * Zi;       // w = Z^(p-k)
            double nwi = wr * Zi + wi * Zr;
            wr = nwr; wi = nwi;
        }
        double ndr = dr * Pr - di * Pi + dcr;
        double ndi = dr * Pi + di * Pr + dci;
        dr = ndr; di = ndi;

        ++m;
        double zr = orbit[2 * m] + dr;
        double zi = orbit[2 * m + 1] + di;
        m2 = zr * zr + zi * zi;
        if (!(m2 <= bail2)) {
            escaped = 1;
            if (!(m2 < 1.0e308)) m2 = bail2 * 4.0;
            break;
        }
        if (check_period) {
            double ar = zr - sr, ai = zi - si;
            if (ar * ar + ai * ai < 1.0e-32) { it = max_iter; break; }
            if (++since >= next_check) { since = 0; next_check <<= 1; sr = zr; si = zi; }
        }
        // Rebasing (Zhuoran): whenever the true orbit is smaller than the delta,
        // or the reference runs out, restart the reference from Z[0].  This is
        // what keeps the perturbation glitch-free.
        double d2 = dr * dr + di * di;
        if (m2 < d2 || m >= olen) { dr = zr; di = zi; m = 0; }
    }
    emit(escaped, it, m2, dzr, dzi, dzexp, log_bail, log_n, step, want_de,
         out_nu, out_de, idx);
}
"""


class MultibrotEngine:
    """Renders one view of the Multibrot set into (smooth iteration, log pixel-distance).

    Two kernels sit behind one `render()` call:

    * below a view width of ~1e-9 float64 still resolves individual pixels, so the
      straightforward iteration is used and any real exponent works;
    * past that the pixel spacing falls under the rounding error of the coordinates
      themselves, so the render switches to perturbation against a reference orbit
      carried in arbitrary precision.  Only the deltas live in float64, and they
      stay representable down to a view width of ~1e-300.
    """

    PERTURB_BELOW = 1e-9          # view width at which float64 coordinates give out

    def __init__(self, exponent, bailout=1e6):
        if not HAVE_GPU:
            raise RuntimeError("No CUDA device — enable a GPU runtime (Runtime > Change runtime type).")
        self.n = float(exponent)
        p = round(self.n)
        self.p = int(p) if abs(self.n - p) < 1e-12 and p >= 2 else 0
        self.bail = float(bailout)
        self.bail2 = self.bail ** 2
        self.log_bail = math.log(self.bail)
        self.log_n = math.log(abs(self.n)) if abs(self.n) > 1.0001 else 0.0
        self._mod = cp.RawModule(code=CUDA_SOURCE)
        self._direct = self._mod.get_function("mb_direct")
        self._perturb = self._mod.get_function("mb_perturb")
        self._binom = (cp.asarray([float(math.comb(self.p, k)) for k in range(self.p + 1)],
                                  dtype=cp.float64) if self.p else None)
        self._cache = None
        # Cycle detection cuts the cost of interior pixels, which dominate deep views.
        # Section 8 checks that it leaves the picture unchanged.
        self.check_period = True

    # -- capability flags ---------------------------------------------------
    @property
    def can_perturb(self):  return self.p >= 2
    @property
    def can_de(self):       return self.p >= 2

    @staticmethod
    def precision_for(span):
        """Decimal digits the reference orbit needs to out-resolve one pixel."""
        return max(30, int(-math.log10(max(float(span), 1e-320))) + 25)

    # -- reference orbit ----------------------------------------------------
    def reference_orbit(self, center, max_iter, dps=None, progress=False):
        """Iterate z -> z^p + c at `dps` decimal digits and hand the orbit to the GPU.

        The whole movie shares one centre, so this is computed once per render and
        reused for every frame.  Returns (device array, usable length).
        """
        if not self.can_perturb:
            raise RuntimeError("Perturbation needs a whole exponent >= 2.")
        dps = dps or self.precision_for(1e-30)
        key = (mpmath.nstr(center.real, 45), mpmath.nstr(center.imag, 45), dps, self.p)
        c = self._cache
        if c and c["key"] == key and (c["escaped"] or c["len"] >= max_iter):
            return c["dev"], c["len"]

        prev, mpmath.mp.dps = mpmath.mp.dps, dps
        try:
            re = np.zeros(max_iter + 1); im = np.zeros(max_iter + 1)
            Z, p, olen, escaped = mpmath.mpc(0, 0), self.p, max_iter, False
            it = tqdm(range(max_iter), desc="reference orbit", leave=False) if progress else range(max_iter)
            for i in it:
                w, b, e = mpmath.mpc(1, 0), Z, p      # binary exponentiation
                while e:
                    if e & 1: w *= b
                    b *= b; e >>= 1
                Z = w + center
                zr = float(Z.real); zi = float(Z.imag)
                re[i + 1] = zr; im[i + 1] = zi
                if zr * zr + zi * zi > 4.0:           # the reference itself escaped
                    olen, escaped = i + 1, True
                    break
        finally:
            mpmath.mp.dps = prev

        flat = np.empty(2 * (olen + 1), dtype=np.float64)
        flat[0::2] = re[:olen + 1]; flat[1::2] = im[:olen + 1]
        dev = cp.asarray(flat)
        self._cache = dict(key=key, dev=dev, len=olen, escaped=escaped)
        if escaped and olen < max_iter // 4:
            print(f"warning · the reference point escapes after {olen} iterations, so most "
                  f"pixels fall back to plain float64.  Pick a centre inside the set.")
        return dev, olen

    # -- rendering ----------------------------------------------------------
    def render(self, center, span, width, height, max_iter, rotation=0.0,
               want_de=None, mode="auto", ref=None, out=None, check_period=None):
        """Render one frame.  `span` is the width of the view in complex-plane units."""
        span = float(span)
        step = span / width
        ca, sa = math.cos(rotation), math.sin(rotation)
        want_de = self.can_de if want_de is None else bool(want_de) and self.can_de
        n_px = width * height
        nu, de = out if out is not None else (cp.empty(n_px, cp.float32), cp.empty(n_px, cp.float32))

        use_perturb = (mode == "perturb") or (mode == "auto" and self.can_perturb and span < self.PERTURB_BELOW)
        if use_perturb and not self.can_perturb:
            raise RuntimeError("Perturbation needs a whole exponent >= 2.")

        threads = 128
        blocks = (n_px + threads - 1) // threads
        if check_period is None:
            check_period = self.check_period
        common = (np.int32(width), np.int32(height), np.int32(max_iter),
                  np.float64(self.bail2), np.float64(self.log_bail), np.float64(self.log_n),
                  np.int32(want_de), np.int32(check_period), nu, de)

        if use_perturb:
            orbit, olen = ref if ref is not None else self.reference_orbit(
                center, max_iter, self.precision_for(span))
            self._perturb((blocks,), (threads,),
                          (orbit, np.int32(olen), self._binom, np.int32(self.p),
                           np.float64(0.0), np.float64(0.0),
                           np.float64(step), np.float64(ca), np.float64(sa)) + common)
        else:
            self._direct((blocks,), (threads,),
                         (np.float64(float(center.real)), np.float64(float(center.imag)),
                          np.float64(step), np.float64(ca), np.float64(sa),
                          np.float64(self.n), np.int32(self.p)) + common)
        return nu.reshape(height, width), de.reshape(height, width)


ENGINE = MultibrotEngine(CFG.exponent) if HAVE_GPU else None
print(f"engine ready · exponent {CFG.exponent:g} · "
      f"perturbation {'yes' if ENGINE and ENGINE.can_perturb else 'no'} · "
      f"distance estimator {'yes' if ENGINE and ENGINE.can_de else 'no'}")

## 4 · Colour

Escape counts climb as the dive deepens, so anything keyed to an absolute maximum —
the `LogNorm(vmin=1, vmax=max_iter)` the original notebook used — drifts and flickers
as the frames go by. Instead the palette is cyclic and indexed by a running function
of the smooth escape count, so bands flow outward at a steady rate however deep the
view goes.

Blending and supersampled downsampling both happen in linear light, with the gamma
encode applied once at the end; interpolating colours in sRGB is what makes hand-rolled
palettes look muddy in the midtones.

`DE_SHADING` uses the distance estimator to darken pixels near the boundary. It is the
single biggest quality win at depth — it anti-aliases the edge and keeps filaments
visible when they are thinner than a pixel.

In [ ]:
#@title 4 · Colour — cyclic palettes, distance shading, downsampling

# Cyclic palettes: the last colour flows back into the first, so the bands keep
# moving smoothly however deep the escape counts climb.
PALETTES = {
    "ember":    ["#04010a", "#2b0733", "#7a1140", "#d2401f", "#f79d3c", "#ffe9b0",
                 "#f79d3c", "#7a1140"],
    "aurora":   ["#02040f", "#062b3f", "#0d7a6e", "#5fd39b", "#d7f9c8", "#7fb3ff",
                 "#2b3a8f", "#0a0b2a"],
    "twilight": ["#1b1b3a", "#4a3a8c", "#9a6fb0", "#e0a3a3", "#f6e2c8", "#a3c4e0",
                 "#4a6fb0", "#20214a"],
    "goldleaf": ["#0a0700", "#3a2400", "#8a5a08", "#d9a021", "#ffe08a", "#fffbe8",
                 "#d9a021", "#4a2f04"],
    "abyss":    ["#000308", "#001b2e", "#01426a", "#1a8fb8", "#8fe3e8", "#e8fbff",
                 "#2a6fa8", "#00121f"],
    "neon":     ["#08000f", "#3d0a52", "#a112b0", "#ff3d9a", "#ffd166", "#3ef2b5",
                 "#1f6ef2", "#20064a"],
    "copper":   ["#0b0402", "#3b1608", "#8a3b18", "#c9702f", "#f0b27a", "#fdf1e0",
                 "#a3502a", "#2a0f06"],
    "pastel":   ["#ffb3ba", "#ffdfba", "#ffffba", "#baffc9", "#bae1ff", "#e6e6fa",
                 "#f7c8e0", "#d6c8ff"],
}


def _hex_to_linear(h):
    """sRGB hex -> linear light, so blends and box-filtering behave physically."""
    h = h.lstrip("#")
    v = np.array([int(h[i:i + 2], 16) for i in (0, 2, 4)], dtype=np.float64) / 255.0
    return v ** 2.2


def make_palette(colors, n=4096):
    """Build a cyclic lookup table in linear light with smoothstep blending."""
    if isinstance(colors, str):
        colors = PALETTES[colors]
    a = np.array([_hex_to_linear(c) for c in colors])
    a = np.vstack([a, a[:1]])                       # wrap the last stop into the first
    k = len(a) - 1
    x = np.linspace(0.0, k, n, endpoint=False)
    i0 = np.floor(x).astype(int)
    f = (x - i0)[:, None]
    f = f * f * (3.0 - 2.0 * f)
    lut = a[i0] * (1.0 - f) + a[i0 + 1] * f
    return cp.asarray(lut, dtype=cp.float32)


def colorize(nu, de, palette, cfg, phase=0.0):
    """Map (smooth iteration, log pixel-distance) onto linear-light RGB."""
    interior = nu < 0.0
    v = cp.maximum(nu, 0.0)
    if cfg.color_mode == "log":
        t = cp.log(v + cfg.color_offset)
    elif cfg.color_mode == "sqrt":
        t = cp.sqrt(v)
    else:
        t = v
    t = t * cfg.color_density + phase
    t -= cp.floor(t)                                # wrap into [0,1)
    n_lut = palette.shape[0]
    idx = cp.minimum((t * n_lut).astype(cp.int32), n_lut - 1)
    rgb = palette[idx]

    if cfg.de_shading and de is not None:
        # `de` is log(distance in pixels).  Near the boundary the distance collapses,
        # which is exactly where the filaments are, so this both anti-aliases the
        # edge and makes fine structure legible at any depth.
        d = cp.exp(cp.clip(de, -50.0, 20.0))
        shade = d / (d + cfg.de_softness)
        rgb = rgb * (cfg.de_floor + (1.0 - cfg.de_floor) * shade)[..., None]

    if cfg.exposure != 1.0:
        rgb = rgb * cfg.exposure
    rgb[interior] = cp.asarray(_hex_to_linear(cfg.interior_color), dtype=cp.float32)
    return rgb


def to_uint8(rgb, supersample=1, gamma=2.2):
    """Box-filter the supersampled buffer in linear light, then gamma-encode."""
    if supersample > 1:
        s = supersample
        h, w = rgb.shape[0] // s, rgb.shape[1] // s
        rgb = cp.ascontiguousarray(rgb[:h * s, :w * s]).reshape(h, s, w, s, 3).mean(axis=(1, 3))
    out = cp.clip(rgb, 0.0, 1.0) ** (1.0 / gamma)
    return (out * 255.0 + 0.5).astype(cp.uint8)


PALETTE_LUT = make_palette(CFG.palette) if HAVE_GPU else None
print(f"palette · {CFG.palette} · {len(PALETTES[CFG.palette])} stops · "
      f"{CFG.color_mode} index, density {CFG.color_density:g}, "
      f"{CFG.color_cycles:g} cycles over the movie")

## 5 · Camera path

Zoom is exponential in *eased* time rather than in time itself. A raw exponential
opens like a jump cut and ends as if the file were truncated; easing in and out gives
the shot a settle at each end while holding a constant number of e-folds per second
through the middle.

The iteration budget grows with depth on the same schedule, monotonically — letting it
drop between frames would make interior regions pop in and out.

In [ ]:
#@title 5 · Camera path — easing, iteration schedule, colour drift

def smoothstep(x):
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)


def build_path(cfg):
    """Everything that varies frame to frame, precomputed as arrays.

    Zoom is exponential in *eased* time rather than in time itself: the shot opens
    slowly, settles into a constant number of e-folds per second, and eases out at
    the end.  A raw exponential start looks like a jump cut, and a raw exponential
    finish looks like the movie was truncated.
    """
    n_move = max(2, int(round(cfg.duration * cfg.fps)))
    n_hold = int(round(cfg.hold_end * cfg.fps))

    t = np.linspace(0.0, 1.0, n_move)
    v = np.ones(n_move)
    if cfg.ease_in > 0:  v *= smoothstep(t / cfg.ease_in)
    if cfg.ease_out > 0: v *= smoothstep((1.0 - t) / cfg.ease_out)
    v = np.maximum(v, 1e-9)
    s = np.cumsum(v); s -= s[0]
    s /= s[-1]                                     # 0 -> 1, monotone
    if n_hold:
        s = np.concatenate([s, np.ones(n_hold)])

    spans = cfg.start_span * (cfg.final_span / cfg.start_span) ** s
    rotation = 2.0 * np.pi * cfg.rotation_turns * s
    depth = np.log10(cfg.start_span / spans)       # decades of magnification so far
    iters = np.clip(cfg.base_iter + cfg.iter_scale * depth ** cfg.iter_power,
                    cfg.base_iter, cfg.iter_cap).astype(np.int64)
    iters = np.maximum.accumulate(iters)           # never let the budget drop
    phase = cfg.color_phase + np.linspace(0.0, cfg.color_cycles, len(s))
    return dict(spans=spans, rotation=rotation, iters=iters, phase=phase,
                progress=s, n_move=n_move, n_hold=n_hold)


PATH = build_path(CFG)
_ef = math.log(CFG.start_span / CFG.final_span)
print(f"path · {len(PATH['spans'])} frames"
      f"  ({PATH['n_move']} moving + {PATH['n_hold']} held)\n"
      f"       {_ef:.1f} e-folds total, {_ef / max(CFG.duration, 1e-9):.3f} per second "
      f"(one doubling every {0.6931 * CFG.duration / max(_ef, 1e-9):.1f}s)\n"
      f"       max_iter {PATH['iters'][0]} -> {PATH['iters'][-1]}")

if not HAVE_GPU:
    print("\n(no GPU — the plot below still works, rendering does not)")

fig, ax = plt.subplots(1, 3, figsize=(13, 2.8))
_x = np.arange(len(PATH["spans"])) / CFG.fps
ax[0].semilogy(_x, PATH["spans"]); ax[0].set_title("view width"); ax[0].set_xlabel("seconds")
ax[1].plot(_x, PATH["iters"]);     ax[1].set_title("max_iter")
ax[2].plot(_x, -np.gradient(np.log(PATH["spans"])) * CFG.fps)
ax[2].set_title("zoom rate (e-folds/s)")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6 · One frame

`render_view` is the whole pipeline for a single image: iterate, colour, downsample,
gamma. The preview below draws a frame from anywhere along the dive so you can judge
palette and detail before committing an hour of GPU time, and 6b renders a still much
larger than video resolution.

In [ ]:
#@title 6 · Render a single view (also used for stills and previews) { display-mode: "form" }

def render_view(center, span, width, height, max_iter, cfg=None, palette=None,
                rotation=0.0, supersample=None, ref=None, mode="auto"):
    """Render one finished RGB frame as a uint8 array of shape (height, width, 3)."""
    cfg = cfg or CFG
    palette = PALETTE_LUT if palette is None else palette
    ss = cfg.supersample if supersample is None else supersample
    nu, de = ENGINE.render(center, span, width * ss, height * ss, int(max_iter),
                           rotation=rotation, want_de=cfg.de_shading, ref=ref, mode=mode)
    rgb = colorize(nu, de if cfg.de_shading else None, palette, cfg,
                   phase=cfg.color_phase)
    return cp.asnumpy(to_uint8(rgb, ss, cfg.gamma))


def iterations_for(cfg, span):
    """The same iteration budget the movie would use at this view width."""
    depth = math.log10(cfg.start_span / max(float(span), 1e-320))
    return int(np.clip(cfg.base_iter + cfg.iter_scale * max(depth, 0.0) ** cfg.iter_power,
                       cfg.base_iter, cfg.iter_cap))


def show(img, title=None, size=8):
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(size, size * h / w))
    ax.imshow(img); ax.axis("off")
    if title:
        ax.set_title(title, color="white", fontsize=11)
    fig.patch.set_facecolor("#0b0b10")
    plt.tight_layout(); plt.show()


#@markdown Renders one frame from anywhere along the dive so you can judge colour and
#@markdown detail before committing to a full render.  `1.0` is the last frame.
PREVIEW_AT = 1.0  #@param {type:"slider", min:0, max:1, step:0.01}
PREVIEW_WIDTH = 900  #@param {type:"number"}

if HAVE_GPU:
    _i = min(len(PATH["spans"]) - 1, int(PREVIEW_AT * (len(PATH["spans"]) - 1)))
    _span, _iter = PATH["spans"][_i], int(PATH["iters"][_i])
    _h = int(PREVIEW_WIDTH * CFG.height / CFG.width) // 2 * 2
    _t0 = time.time()
    _img = render_view(CFG.center, _span, PREVIEW_WIDTH, _h, _iter,
                       rotation=float(PATH["rotation"][_i]))
    cp.cuda.Stream.null.synchronize()
    show(_img, f"t = {_i / CFG.fps:.0f}s   width {_span:.3e}   "
               f"{math.log10(CFG.start_span / _span):.1f} decades   max_iter {_iter}   "
               f"({time.time() - _t0:.2f}s)")

In [ ]:
#@title 6b · Poster still — one frame at full quality { display-mode: "form" }

#@markdown Renders a single frame much larger than video resolution, for a print or a
#@markdown thumbnail.  `STILL_ZOOM` is in decades, like `FINAL_ZOOM`.
RENDER_STILL = False  #@param {type:"boolean"}
STILL_WIDTH = 3000  #@param {type:"number"}
STILL_ZOOM = 40  #@param {type:"number"}
STILL_SUPERSAMPLE = 3  #@param [1, 2, 3, 4] {type:"raw"}
STILL_ROTATION_TURNS = 0.0  #@param {type:"number"}

if HAVE_GPU and RENDER_STILL:
    _w = int(STILL_WIDTH) // 2 * 2
    _h = int(_w * CFG.height / CFG.width) // 2 * 2
    _span = CFG.start_span / 10.0 ** float(STILL_ZOOM)
    _it = iterations_for(CFG, _span)
    _ref = None
    if ENGINE.can_perturb and _span < ENGINE.PERTURB_BELOW:
        _ref = ENGINE.reference_orbit(CFG.center, _it, ENGINE.precision_for(_span),
                                      progress=True)
    print(f"rendering {_w}x{_h} at {STILL_SUPERSAMPLE}x{STILL_SUPERSAMPLE} supersampling "
          f"({_w * _h * STILL_SUPERSAMPLE ** 2 / 1e6:.0f} Mpx), max_iter {_it} ...")
    _t0 = time.time()
    _img = render_view(CFG.center, _span, _w, _h, _it,
                       rotation=2 * math.pi * float(STILL_ROTATION_TURNS),
                       supersample=int(STILL_SUPERSAMPLE), ref=_ref)
    _out = Path(CFG.outdir); _out.mkdir(parents=True, exist_ok=True)
    _p = _out / f"multibrot_z{CFG.exponent:g}_1e{STILL_ZOOM:g}x_{_w}x{_h}.png"
    plt.imsave(_p, _img)
    print(f"{_p}  ({time.time() - _t0:.1f}s, {_p.stat().st_size / 2**20:.1f} MB)")
    show(_img, size=9)

## 7 · Where to dive

A deep zoom is only worth watching if the centre keeps producing new structure all the
way down. The curated coordinates below were found by descent search and carry the
depth each was verified to; `find_deep_target` runs the same search on the GPU to go
further, or to find somewhere new.

The search recentres on a point *inside* the set next to the deepest escaping pixel it
can find. Staying inside matters for more than composition: the reference orbit has to
survive the entire iteration budget, and a centre that escapes early degrades the
perturbation engine back toward plain float64.

The contact sheet is the honest check — if the last few thumbnails are flat, the dive
bottoms out and you want a different target or a smaller `FINAL_ZOOM`.

In [ ]:
#@title 7 · Targets — curated coordinates and an automatic search { display-mode: "form" }

def _span_precision(span):
    """Decimal digits needed to address a view this narrow."""
    return max(30, int(-mpmath.log10(span)) + 25)


def _interior_density(mask, r=3):
    """Fraction of each (2r+1)^2 neighbourhood that is inside the set."""
    m = mask.astype(np.float32)
    ii = np.pad(m, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
    H, W = m.shape
    y0 = np.clip(np.arange(H) - r, 0, H); y1 = np.clip(np.arange(H) + r + 1, 0, H)
    x0 = np.clip(np.arange(W) - r, 0, W); x1 = np.clip(np.arange(W) + r + 1, 0, W)
    tot = (ii[np.ix_(y1, x1)] - ii[np.ix_(y0, x1)]
           - ii[np.ix_(y1, x0)] + ii[np.ix_(y0, x0)])
    return tot / ((y1 - y0)[:, None] * (x1 - x0)[None, :]).astype(np.float32)


def _rank_candidates(nu, shrink, limit=32):
    """Interior pixels worth recentring on, best first.

    Two hard requirements, not preferences.  The centre has to be inside the set, or
    the reference orbit escapes and the deep engine degrades back to plain float64.
    And it has to be close enough to the boundary that the boundary survives the next
    zoom step: with a shrink of s the next frame covers 1/s of this one, so an exterior
    pixel must lie within grid/(2s) of the candidate.  Merely *preferring* solid
    interior points is what makes a descent wander into the middle of a component and
    bottom out — the dive then ends up staring at a flat region, which is the failure
    `verify_target` below exists to catch.
    """
    interior = nu < 0.0
    exterior = ~interior
    if not (interior.any() and exterior.any()):
        return [], 0.0, 0.0
    grid = max(nu.shape)
    reach = max(1, int(grid / (2.0 * shrink)))   # a pre-filter; the caller verifies
    ey, ex = np.unravel_index(np.argmax(np.where(exterior, nu, -np.inf)), nu.shape)
    keeps_boundary = interior & (_interior_density(interior, reach) < 1.0)
    solid = _interior_density(interior, 2) >= 0.5          # not a lone speck
    for mask in (keeps_boundary & solid, keeps_boundary, interior & solid, interior):
        iy, ix = np.nonzero(mask)
        if iy.size:
            break
    order = np.argsort((iy - ey) ** 2 + (ix - ex) ** 2)[:limit]
    return ([(int(iy[k]), int(ix[k])) for k in order],
            float(nu[exterior].max()), float(interior.mean()))


def find_deep_target(engine, seed_re, seed_im, final_zoom, start_span=1.2,
                     grid=384, shrink=4.0, base_iter=2500, iter_growth=0.35,
                     tries=8, verbose=True):
    """Walk down to a point that still grows new structure at `final_zoom` decades.

    Each level renders the current view small, ranks the interior pixels next to the
    deepest escaping one, and takes the best candidate whose reference orbit actually
    survives the *next* level's iteration budget — checking rather than assuming,
    because orbit survival is the thing that decides whether the deep engine works at
    all.  The centre is carried in arbitrary precision the whole way down.
    """
    prec = max(60, int(final_zoom) + 40)
    mpmath.mp.dps = prec
    center = mpmath.mpc(mpmath.mpf(str(seed_re)), mpmath.mpf(str(seed_im)))
    span = mpmath.mpf(str(start_span))
    target_span = mpmath.mpf(str(start_span)) / mpmath.mpf(10) ** final_zoom

    level, stalled = 0, 0
    while span > target_span:
        max_iter = int(min(400000, base_iter * (1.0 + iter_growth * level)))
        nxt = int(min(400000, base_iter * (1.0 + iter_growth * (level + 1))))
        dps = max(40, _span_precision(span))
        mpmath.mp.dps = max(prec, dps)

        ref = (engine.reference_orbit(center, max_iter, dps)
               if engine.can_perturb and float(span) < engine.PERTURB_BELOW else None)
        nu, _ = engine.render(center, float(span), grid, grid, max_iter,
                              want_de=False, ref=ref)
        cands, deepest, ifrac = _rank_candidates(cp.asnumpy(nu), shrink)
        if not cands:
            # Entirely inside or entirely outside: give the budget room and retry.
            stalled += 1
            if stalled > 3:
                if verbose: print(f"  level {level}: no boundary in view, stopping here")
                break
            base_iter = int(base_iter * 1.8)
            continue
        stalled = 0

        # Try candidates in order of proximity to the boundary and take the first that
        # passes both hard checks: its reference orbit survives the next level's budget,
        # and the next level's view still contains a boundary.  The second check is made
        # by rendering that view rather than estimating it -- a heuristic about how close
        # is close enough is exactly the thing that goes wrong here, and the result is a
        # coordinate that looks fine at the bottom but spends the middle of the dive
        # inside a solid region.
        chosen, survived, best = None, 0, None
        nspan = span / mpmath.mpf(str(shrink))
        for ty, tx in cands[:tries]:
            fx = (tx - (grid - 1) / 2.0) / grid          # small floats ...
            fy = (ty - (grid - 1) / 2.0) / grid
            trial = center + mpmath.mpc(mpmath.mpf(fx) * span,   # ... scaled exactly
                                        -mpmath.mpf(fy) * span)
            # The orbit only has to survive once perturbation is actually in use;
            # above that the direct kernel does not consult it, and computing one per
            # candidate at every shallow level is the bulk of the search's cost.
            if engine.can_perturb and float(nspan) < engine.PERTURB_BELOW:
                tref, olen2 = engine.reference_orbit(trial, nxt,
                                                     max(40, _span_precision(nspan)))
            else:
                tref, olen2 = None, nxt
            nnu, _ = engine.render(trial, float(nspan), 96, 96, nxt,
                                   want_de=False, ref=tref)
            inside = float((cp.asnumpy(nnu) < 0).mean())
            score = (olen2 >= nxt, 0.01 < inside < 0.99)
            if best is None or score > best[2]:
                best = (trial, olen2, score)
            if all(score):
                chosen, survived = trial, olen2
                break
        if chosen is None:                                # nothing perfect; take the best
            chosen, survived = best[0], best[1]
        center = chosen

        if verbose:
            print(f"  level {level:3d}: width 1e{float(mpmath.log10(span)):7.2f}  "
                  f"max_iter {max_iter:6d}  deepest {deepest:8.0f}  "
                  f"interior {ifrac * 100:5.1f}%  orbit {survived}/{nxt}"
                  f"{'' if survived >= nxt else '  <-- reference escapes'}")
        span /= mpmath.mpf(str(shrink))
        level += 1

    mpmath.mp.dps = prec
    return center, span


def verify_target(engine, cfg, center=None, grid=384, samples=6):
    """Does this centre keep producing structure all the way down?

    Checking only the final frame is not enough: a coordinate can be alive at the
    bottom and still spend the middle of the dive inside a solid region, which is five
    minutes of watching a flat colour.  This samples the whole path.
    """
    center = cfg.center if center is None else center
    depths = np.linspace(cfg.final_zoom / samples, cfg.final_zoom, samples)
    worst, bad = [], 0
    ref = None
    if engine.can_perturb:
        mi = iterations_for(cfg, cfg.final_span)
        ref = engine.reference_orbit(center, mi, engine.precision_for(cfg.final_span),
                                     progress=True)
        if ref[1] < mi:
            print(f"  warning · the reference orbit escapes after {ref[1]}/{mi} iterations, "
                  f"so deep frames lose precision. Re-run the search below.")
    for d in depths:
        span = cfg.start_span / 10.0 ** float(d)
        use_ref = ref if (ref is not None and span < engine.PERTURB_BELOW) else None
        nu, _ = engine.render(center, span, grid, grid, iterations_for(cfg, span),
                              want_de=False, ref=use_ref)
        nu = cp.asnumpy(nu)
        inside = float((nu < 0).mean())
        flat = inside > 0.99 or inside < 0.01
        bad += flat
        worst.append((d, inside))
        print(f"  1e{d:5.1f}x   interior {inside * 100:5.1f}%" + ("   <-- featureless" if flat else ""))
    if bad:
        print(f"\n  {bad} of {samples} sampled depths are featureless. The dive bottoms out "
              f"or passes through a solid region.\n  Lower FINAL_ZOOM, or set RUN_SEARCH "
              f"below to find a coordinate that stays alive this deep.")
    else:
        print("\n  looks good — structure at every depth sampled.")
    return worst, bad == 0


def contact_sheet(center, cfg, decades, width=340, cols=4, palette=None):
    """Thumbnails down the dive, so you can see what the movie will actually show."""
    rows = (len(decades) + cols - 1) // cols
    h = int(width * cfg.height / cfg.width) // 2 * 2
    fig, axes = plt.subplots(rows, cols, figsize=(3.1 * cols, 3.1 * rows * h / width))
    axes = np.atleast_1d(axes).ravel()
    deepest = cfg.start_span / 10.0 ** max(decades)
    ref = None
    if ENGINE.can_perturb:
        ref = ENGINE.reference_orbit(center, iterations_for(cfg, deepest),
                                     ENGINE.precision_for(deepest), progress=True)
    for ax, d in zip(axes, decades):
        span = cfg.start_span / 10.0 ** d
        use_ref = ref if span < ENGINE.PERTURB_BELOW else None
        img = render_view(center, span, width, h, iterations_for(cfg, span),
                          cfg=cfg, palette=palette, supersample=1, ref=use_ref)
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"1e{d:g}x", color="white", fontsize=9)
    for ax in axes[len(decades):]:
        ax.axis("off")
    fig.patch.set_facecolor("#0b0b10")
    plt.tight_layout(); plt.show()


#@markdown Set `TARGET = "auto"` in section 2 and run this to search for a fresh
#@markdown coordinate, then paste the printed values back into `CENTER_RE` / `CENTER_IM`.
#@markdown The seed only has to be somewhere in the first frame; `0, 0` works for any
#@markdown exponent because the whole set is in view at the start.
RUN_SEARCH = False  #@param {type:"boolean"}
SEARCH_SEED_RE = "0"  #@param {type:"string"}
SEARCH_SEED_IM = "0"  #@param {type:"string"}
VERIFY_TARGET = True  #@param {type:"boolean"}
SHOW_CONTACT_SHEET = True  #@param {type:"boolean"}

if HAVE_GPU and RUN_SEARCH:
    print(f"searching for a z^{CFG.exponent:g} target at 1e{CFG.final_zoom:g}x ...")
    _c, _s = find_deep_target(ENGINE, SEARCH_SEED_RE, SEARCH_SEED_IM, CFG.final_zoom,
                              start_span=CFG.start_span)
    mpmath.mp.dps = CFG.precision
    _re = mpmath.nstr(_c.real, CFG.precision - 5, strip_zeros=False)
    _im = mpmath.nstr(_c.imag, CFG.precision - 5, strip_zeros=False)
    print(f"\nCENTER_RE = \"{_re}\"\nCENTER_IM = \"{_im}\"\n")
    CFG.center_re, CFG.center_im = _re, _im
    ENGINE._cache = None
    print("CFG updated in place — re-run section 5 onwards, or paste the strings above "
          "into section 2 to keep them.")

if HAVE_GPU and VERIFY_TARGET:
    print(f"checking the dive to 1e{CFG.final_zoom:g}x ...")
    _worst, TARGET_OK = verify_target(ENGINE, CFG)

if HAVE_GPU and SHOW_CONTACT_SHEET:
    _d = sorted({round(x, 2) for x in np.linspace(0.5, CFG.final_zoom, 8)})
    contact_sheet(CFG.center, CFG, _d)

## 8 · Self-test

Worth thirty seconds before an hour-long render. The important check is the first one:
at a view width where float64 is still exact, the perturbation engine and the direct
engine must agree pixel for pixel. The second check is its mirror — at `1e-24` the
direct kernel must collapse into a flat field while perturbation keeps resolving
detail, which is what confirms the deep engine is doing real work rather than
returning something plausible.

In [ ]:
#@title 8 · Self-test — check the kernels before spending an hour on a render

def _test_site(engine, decades, start_span=4.0, grid=192):
    """A centre where the boundary is genuinely in frame at `start_span/10**decades`.

    A test that renders a patch with no boundary in it compares one flat field against
    another and passes without checking anything, so the site is found by the same
    descent the target search uses rather than assumed.
    """
    c, span = find_deep_target(engine, "0", "0", decades, start_span=start_span,
                               grid=grid, base_iter=1500, verbose=False)
    return c, float(span)


def self_test(engine, cfg):
    rows = []

    def record(name, ok, detail):
        rows.append(bool(ok))
        print(f"  [{'PASS' if ok else 'FAIL'}]  {name:<46s} {detail}")

    def compare(name, a, b, cls_tol=0.999, tol=1e-3, min_escaping=0.02):
        """Two renders of the same view must describe the same picture.

        Compared on the median, not the worst pixel: escape counts for pixels
        asymptotically on the boundary are genuinely chaotic, and a single-ulp
        difference in arithmetic ordering moves them by hundreds of iterations.  A
        frame with almost nothing escaping is reported as a failed test rather than
        a passed one — there would be nothing in it to disagree about.
        """
        a, b = cp.asnumpy(a), cp.asnumpy(b)
        ext = (a >= 0) & (b >= 0)
        if ext.mean() < min_escaping:
            record(name, False, f"only {ext.mean() * 100:.1f}% of the frame escapes — "
                                f"no boundary in view, so the test would be vacuous")
            return
        same = float(((a < 0) == (b < 0)).mean())
        d = np.abs(a[ext] - b[ext])
        record(name, same > cls_tol and float(np.median(d)) < tol,
               f"{same * 100:.3f}% same class, median dnu {np.median(d):.1e}, "
               f"p99 {np.percentile(d, 99):.1e}, {ext.mean() * 100:.0f}% escaping")

    print("self-test  (finding a test site ...)")
    c, span = _test_site(engine, 6.0)          # boundary in frame at ~4e-6
    N, MI = 512, 8000
    print(f"  site: view width {span:.2e}, {MI} iterations\n")

    # 1. Where float64 is still exact the two engines must describe the same picture,
    #    so a disagreement here is a bug in the perturbation kernel, not precision.
    if engine.can_perturb:
        ref = engine.reference_orbit(c, MI, engine.precision_for(span))
        a, _ = engine.render(c, span, N, N, MI, want_de=False, mode="direct")
        b, _ = engine.render(c, span, N, N, MI, want_de=False, mode="perturb", ref=ref)
        compare("perturbation matches direct float64", a, b)

        # 2. Cycle detection skips interior pixels early.  It is a large speed-up on
        #    deep views, which are mostly interior, and must not move a single pixel.
        for mode, kw in (("direct", {}), ("perturb", {"ref": ref})):
            x, _ = engine.render(c, span, N, N, MI, want_de=False, mode=mode,
                                 check_period=False, **kw)
            y, _ = engine.render(c, span, N, N, MI, want_de=False, mode=mode,
                                 check_period=True, **kw)
            x, y = cp.asnumpy(x), cp.asnumpy(y)
            esc = float((x >= 0).mean())
            # An all-interior frame cannot reveal the failure this test exists to
            # catch -- pixels wrongly called interior -- so treat it as a failure.
            record(f"cycle detection leaves the {mode} kernel exact",
                   esc >= 0.02 and np.array_equal(x < 0, y < 0)
                   and float(np.abs(x - y).max()) == 0.0,
                   f"identical classes {np.array_equal(x < 0, y < 0)}, "
                   f"max dnu {np.abs(x - y).max():.1e}, {esc * 100:.0f}% escaping")

    # 3. The mirror of test 1, at the depth this movie ends on: the deep engine must
    #    resolve structure where the direct kernel cannot.
    if engine.can_perturb and cfg.final_span < engine.PERTURB_BELOW:
        mi = iterations_for(cfg, cfg.final_span)
        dref = engine.reference_orbit(cfg.center, mi,
                                      engine.precision_for(cfg.final_span), progress=True)
        pb, _ = engine.render(cfg.center, cfg.final_span, N, N, mi,
                              want_de=False, mode="perturb", ref=dref)
        pd, _ = engine.render(cfg.center, cfg.final_span, N, N, mi,
                              want_de=False, mode="direct")
        pb, pd = cp.asnumpy(pb), cp.asnumpy(pd)
        esc = float((pb >= 0).mean())
        deep = float(np.std(pb[pb >= 0])) if (pb >= 0).any() else 0.0
        flat = float(np.std(pd[pd >= 0])) if (pd >= 0).any() else 0.0
        if esc < 0.01:
            record(f"perturbation resolves at 1e{cfg.final_zoom:g}x", False,
                   f"the last frame is {100 - esc * 100:.1f}% interior — this is the "
                   f"target, not the kernel; re-run section 7")
        else:
            record(f"perturbation resolves at 1e{cfg.final_zoom:g}x", deep > 1.0,
                   f"escape-count spread {deep:.1f} (direct float64: {flat:.1f}), "
                   f"{esc * 100:.0f}% escaping")
        record("reference orbit survives the iteration budget", dref[1] >= mi,
               f"{dref[1]}/{mi} iterations")

    # 4. Smooth escape counts.  Were the fractional part broken or absent the values
    #    would pile up on the integers, which is what makes bands crawl and strobe.
    nu, de = engine.render(c, span, 1024, 1024, MI)
    nun = cp.asnumpy(nu)
    ext = nun >= 0
    if ext.mean() < 0.02:
        record("escape count is smooth, not quantised", False, "no boundary in view")
    else:
        hist, _ = np.histogram(nun[ext] % 1.0, bins=10, range=(0, 1))
        flatness = hist.min() / max(hist.mean(), 1e-9)
        record("escape count is smooth, not quantised",
               flatness > 0.3 and bool(np.isfinite(nun).all()),
               f"fractional parts flat to {flatness:.2f} (1.0 = perfectly uniform)")

        # 5. The distance estimate has to be finite everywhere it is used.
        if engine.can_de:
            d = cp.asnumpy(de)[ext]
            record("distance estimate is finite", bool(np.isfinite(d).all()),
                   f"log pixel-distance spans [{d.min():.1f}, {d.max():.1f}]")

    # 6. Colour and camera plumbing.
    lut = make_palette(cfg.palette)
    seam = float(cp.abs(lut[0] - lut[-1]).max())
    record("palette wraps without a seam", seam < 0.05,
           f"{lut.shape[0]} entries, seam gap {seam:.3f}")
    p = build_path(cfg)
    ends_ok = (abs(p["spans"][0] / cfg.start_span - 1) < 1e-9 and
               abs(p["spans"][p["n_move"] - 1] / cfg.final_span - 1) < 1e-6 and
               bool((np.diff(p["spans"]) <= 0).all()))
    record("camera path hits both endpoints", ends_ok,
           f"{p['spans'][0]:.3g} -> {p['spans'][p['n_move'] - 1]:.3e}, monotone")

    bad = rows.count(False)
    print(f"\n{len(rows) - bad}/{len(rows)} passed" +
          ("" if not bad else "  <-- do not start a long render until these are fixed"))
    return bad == 0


if HAVE_GPU:
    SELF_TEST_OK = self_test(ENGINE, CFG)

## 9 · The movie pipeline

Frames are written as raw RGB into an `ffmpeg` process and encoded in short segments
that are concatenated at the end. Segments exist because Colab sessions end without
warning: a dropped runtime costs at most the segment in flight, and re-running resumes
from the first missing one. The configuration fingerprint stored alongside them means a
resumed render can never splice frames from two different shots together.

In [ ]:
#@title 9 · Movie pipeline — segmented encoding with resume

def _spawn_encoder(cfg, path, log_path):
    """An ffmpeg process that eats raw RGB frames on stdin."""
    cmd = [FFMPEG, "-y", "-loglevel", "error",
           "-f", "rawvideo", "-pix_fmt", "rgb24",
           "-s", f"{cfg.width}x{cfg.height}", "-r", str(cfg.fps), "-i", "-",
           "-an", "-c:v", "libx264", "-preset", cfg.x264_preset, "-crf", str(cfg.crf)]
    if cfg.x264_tune:
        cmd += ["-tune", cfg.x264_tune]
    cmd += ["-pix_fmt", "yuv420p", "-g", str(cfg.fps * 2), str(path)]
    log = open(log_path, "wb")
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=log, bufsize=0)
    proc._log_file = log          # closed by the caller once ffmpeg has exited
    return proc


def _concat(cfg, n_seg, outdir):
    listing = outdir / "segments.txt"
    listing.write_text("".join(f"file 'seg_{i:04d}.mp4'\n" for i in range(n_seg)))
    final = outdir / (f"multibrot_z{cfg.exponent:g}_{cfg.width}x{cfg.height}"
                      f"_{cfg.final_zoom:g}decades_{cfg.fingerprint()}.mp4")
    r = subprocess.run([FFMPEG, "-y", "-loglevel", "error", "-f", "concat", "-safe", "0",
                        "-i", str(listing), "-c", "copy", "-movflags", "+faststart",
                        str(final)], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError("concat failed:\n" + r.stderr)
    return final


def render_movie(cfg, engine, palette, path=None, resume=True, still_every=0):
    """Render every frame and encode it, in segments that survive a dropped runtime.

    Colab hands out sessions that end without warning, so frames go straight into
    short self-contained .mp4 segments rather than one long pipe or a directory of
    PNGs.  A rerun picks up at the first missing segment; changing the configuration
    changes the fingerprint and forces a clean start, so a resumed render can never
    splice frames from two different shots together.
    """
    if FFMPEG is None:
        raise RuntimeError("ffmpeg is not installed — rerun section 1.")
    path = path or build_path(cfg)
    outdir = Path(cfg.outdir); outdir.mkdir(parents=True, exist_ok=True)
    stills = outdir / "stills"; stills.mkdir(exist_ok=True)

    total = len(path["spans"])
    seg_len = max(1, int(cfg.segment_seconds * cfg.fps))
    n_seg = (total + seg_len - 1) // seg_len
    state_p = outdir / "render_state.json"
    fingerprint = cfg.fingerprint()

    done = 0
    if resume and state_p.exists():
        st = json.loads(state_p.read_text())
        if st.get("fingerprint") == fingerprint:
            done = int(st.get("segments_done", 0))
            # trust only segments that are actually on disk
            while done and not (outdir / f"seg_{done - 1:04d}.mp4").exists():
                done -= 1
            if done:
                print(f"resuming · {done}/{n_seg} segments already encoded "
                      f"({min(done * seg_len, total)}/{total} frames)")
        else:
            print("configuration changed since the last run — starting from frame 0")
            for f in outdir.glob("seg_*.mp4"):
                f.unlink()

    # One centre for the whole movie means one reference orbit for the whole movie.
    ref = None
    if engine.can_perturb and cfg.final_span < engine.PERTURB_BELOW:
        print(f"computing the reference orbit: {int(path['iters'].max())} iterations "
              f"at {engine.precision_for(cfg.final_span)} digits")
        t0 = time.time()
        ref = engine.reference_orbit(cfg.center, int(path["iters"].max()),
                                     engine.precision_for(cfg.final_span), progress=True)
        print(f"  done in {time.time() - t0:.1f}s (orbit length {ref[1]})")

    ss = cfg.supersample
    W, H = cfg.width * ss, cfg.height * ss
    buf = (cp.empty(W * H, cp.float32), cp.empty(W * H, cp.float32))

    bar = tqdm(total=total, initial=min(done * seg_len, total), unit="frame",
               desc="rendering")
    t_start = time.time()
    for seg in range(done, n_seg):
        lo, hi = seg * seg_len, min(total, (seg + 1) * seg_len)
        part = outdir / f"seg_{seg:04d}.mp4"
        log = outdir / f"seg_{seg:04d}.log"
        proc = _spawn_encoder(cfg, part, log)
        try:
            for i in range(lo, hi):
                span = float(path["spans"][i])
                use_ref = ref if (ref is not None and span < engine.PERTURB_BELOW) else None
                nu, de = engine.render(cfg.center, span, W, H, int(path["iters"][i]),
                                       rotation=float(path["rotation"][i]),
                                       want_de=cfg.de_shading, ref=use_ref, out=buf)
                rgb = colorize(nu, de if cfg.de_shading else None, palette, cfg,
                               phase=float(path["phase"][i]))
                frame = cp.asnumpy(to_uint8(rgb, ss, cfg.gamma))
                proc.stdin.write(frame.tobytes())
                if still_every and i % still_every == 0:
                    plt.imsave(stills / f"frame_{i:06d}.png", frame)
                bar.update(1)
        except BaseException:
            proc.kill(); raise
        finally:
            if proc.stdin and not proc.stdin.closed:
                proc.stdin.close()
        rc = proc.wait()
        proc._log_file.close()
        if rc != 0:
            raise RuntimeError(f"ffmpeg failed on segment {seg}:\n{log.read_text()}")
        state_p.write_text(json.dumps(dict(fingerprint=fingerprint, segments_done=seg + 1,
                                           n_segments=n_seg, total_frames=total), indent=1))
    bar.close()

    final = _concat(cfg, n_seg, outdir)
    mb = final.stat().st_size / 2**20
    print(f"\nwrote {final}\n"
          f"  {total} frames · {total / cfg.fps:.0f}s · {cfg.width}x{cfg.height} @ {cfg.fps}fps\n"
          f"  {mb:.1f} MB ({mb * 8 / (total / cfg.fps):.1f} Mbit/s) · "
          f"render took {(time.time() - t_start) / 60:.1f} min")
    return final


print("pipeline loaded · render_movie(CFG, ENGINE, PALETTE_LUT)")

## 10 · Calibrate and benchmark

Two things worth knowing before starting a long render.

**Calibration** sizes the iteration budget to this particular dive. Guessing low is the
usual way a deep zoom goes wrong: pixels that would have escaped get painted as
interior, so solid blobs appear and then pop a few frames later when the budget grows.
The depth alone does not tell you the number — at `1e44x` the shipped `z^8` coordinate
needs around 60,000 iterations — so this measures it.

**The benchmark** times real frames from along the dive. Cost per frame climbs steeply
with depth, so extrapolating from the cheap frames at the start would understate the
total by an order of magnitude. If the estimate is longer than you want to wait, the
levers in descending order of effect are `SUPERSAMPLE` (quadratic), `QUALITY`,
`FINAL_ZOOM`, and `DURATION_SECONDS`.

In [ ]:
#@title 10 · Calibrate and benchmark { display-mode: "form" }

def calibrate_iterations(cfg, engine, probes=4, headroom=1.4, grid=320, verbose=True):
    """Fit the iteration schedule to what this dive actually needs.

    The right budget depends on where you are diving, not just how far, and guessing
    low is the most common way a deep zoom goes wrong: pixels that would have escaped
    are painted as interior, so solid blobs appear and then pop as the budget grows a
    few frames later.  This renders small probe frames with generous budgets, reads
    off the escape count that 99.9% of escaping pixels come in under, and sizes the
    schedule to cover the worst probe with headroom to spare.
    """
    depths = np.linspace(max(1.0, cfg.final_zoom * 0.35), cfg.final_zoom, probes)
    budget0 = int(np.clip(3 * (cfg.base_iter + cfg.iter_scale * cfg.final_zoom ** cfg.iter_power),
                          20000, cfg.iter_cap))
    ref, ref_len = None, 0
    if engine.can_perturb and cfg.final_span < engine.PERTURB_BELOW:
        ref = engine.reference_orbit(cfg.center, budget0,
                                     engine.precision_for(cfg.final_span), progress=True)
        ref_len = ref[1]

    obs_d, obs_i = [], []
    for d in tqdm(depths, desc="calibrating", unit="probe"):
        span = cfg.start_span / 10.0 ** float(d)
        budget, need = budget0, None
        for _ in range(3):
            use_ref = ref if (ref is not None and span < engine.PERTURB_BELOW) else None
            nu, _ = engine.render(cfg.center, span, grid, grid, budget,
                                  want_de=False, ref=use_ref)
            nu = cp.asnumpy(nu)
            ext = nu[nu >= 0]
            if ext.size == 0:
                break
            need = float(np.percentile(ext, 99.9))
            if need < 0.92 * budget:            # the budget was not the binding limit
                break
            budget = int(min(cfg.iter_cap, budget * 2))
            if ref is not None and budget > ref_len:
                ref = engine.reference_orbit(cfg.center, budget,
                                             engine.precision_for(cfg.final_span))
                ref_len = ref[1]
        if need is not None:
            obs_d.append(float(d)); obs_i.append(need)
            if verbose:
                print(f"    1e{d:5.1f}x needs {need:8.0f} iterations "
                      f"(budget {budget}{'' if need < 0.92 * budget else ', still saturated'})")

    if not obs_d:
        print("  no escaping pixels at any probe depth — this dive ends inside a solid\n"
              "  region, so there is nothing to calibrate against. Check section 7.")
        return cfg
    scale = float(np.max((np.array(obs_i) * headroom - cfg.base_iter)
                         / np.maximum(np.array(obs_d), 1e-9)))
    new = replace(cfg, iter_scale=max(scale, 1.0), iter_power=1.0)
    if verbose:
        print(f"  iteration schedule: {cfg.iter_scale:.0f}*depth^{cfg.iter_power:g} -> "
              f"{new.iter_scale:.0f}*depth   (max_iter at the last frame: "
              f"{iterations_for(cfg, cfg.final_span)} -> {iterations_for(new, new.final_span)})")
    return new


def benchmark(cfg, engine, palette, samples=6, verbose=True):
    """Time real frames from along the dive and extrapolate to the whole movie.

    Cost per frame climbs steeply with depth (deeper frames both carry a larger
    iteration budget and spend more of it), so timing one frame at the start would
    understate the total by an order of magnitude.
    """
    path = build_path(cfg)
    total = len(path["spans"])
    idx = np.unique(np.linspace(0, total - 1, samples).astype(int))

    ref = None
    if engine.can_perturb and cfg.final_span < engine.PERTURB_BELOW:
        ref = engine.reference_orbit(cfg.center, int(path["iters"].max()),
                                     engine.precision_for(cfg.final_span), progress=True)

    ss = cfg.supersample
    W, H = cfg.width * ss, cfg.height * ss
    buf = (cp.empty(W * H, cp.float32), cp.empty(W * H, cp.float32))
    times = []
    for i in tqdm(idx, desc="timing", unit="frame"):
        span = float(path["spans"][i])
        use_ref = ref if (ref is not None and span < engine.PERTURB_BELOW) else None
        cp.cuda.Stream.null.synchronize()
        t0 = time.time()
        nu, de = engine.render(cfg.center, span, W, H, int(path["iters"][i]),
                               rotation=float(path["rotation"][i]),
                               want_de=cfg.de_shading, ref=use_ref, out=buf)
        rgb = colorize(nu, de if cfg.de_shading else None, palette, cfg,
                       phase=float(path["phase"][i]))
        frame = cp.asnumpy(to_uint8(rgb, ss, cfg.gamma))
        times.append(time.time() - t0)

    per_frame = np.interp(np.arange(total), idx, times)
    est = per_frame.sum()
    if verbose:
        print(f"\n  {W}x{H} computed per frame ({W * H / 1e6:.1f} Mpx), "
              f"{cfg.width}x{cfg.height} delivered")
        for i, t in zip(idx, times):
            print(f"    frame {i:6d}  t={i / cfg.fps:6.1f}s  "
                  f"1e{math.log10(cfg.start_span / path['spans'][i]):6.1f}x  "
                  f"max_iter {int(path['iters'][i]):7d}   {t:6.2f}s")
        print(f"\n  estimated render time: {est / 60:.0f} min "
              f"({est / 3600:.1f} h) for {total} frames")
        if est > 12 * 3600:
            print("  that is longer than a Colab session — the render resumes where it "
                  "stopped, but consider a lower QUALITY, SUPERSAMPLE=1, or a shorter "
                  "DURATION_SECONDS.")
    return est, per_frame


#@markdown Sizes the iteration budget to this dive, then times real frames from along
#@markdown it and extrapolates.  Calibration changes the frames, so it also invalidates
#@markdown any partly finished render.
CALIBRATE = True  #@param {type:"boolean"}
RUN_BENCHMARK = True  #@param {type:"boolean"}

if HAVE_GPU and CALIBRATE:
    CFG = calibrate_iterations(CFG, ENGINE)
    PATH = build_path(CFG)

if HAVE_GPU and RUN_BENCHMARK:
    EST_SECONDS, _per_frame = benchmark(CFG, ENGINE, PALETTE_LUT)

## 11 · Render

Interruptible. Re-run this cell after a disconnect and it continues from the last
finished segment.

In [ ]:
#@title 11 · Render the movie { display-mode: "form" }

#@markdown Safe to interrupt and re-run: finished segments are kept and the render
#@markdown picks up where it stopped.  Changing anything in section 2 starts a fresh
#@markdown render instead of splicing mismatched frames together.
SAVE_STILL_EVERY = 300  #@param {type:"number"}

if HAVE_GPU:
    MOVIE_PATH = render_movie(CFG, ENGINE, PALETTE_LUT,
                              still_every=int(SAVE_STILL_EVERY) or 0)
else:
    raise RuntimeError("No GPU — enable one under Runtime > Change runtime type.")

## 12 · Export

The container is deleted when the session ends, so copy the file to Drive or download
it before you close the tab.

In [ ]:
#@title 12 · Watch it, save it, keep it { display-mode: "form" }

#@markdown A Colab container is deleted when the session ends, so copy the file out.
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_FOLDER = "/content/drive/MyDrive/multibrot"  #@param {type:"string"}
DOWNLOAD = False  #@param {type:"boolean"}
PREVIEW_SECONDS = 20  #@param {type:"number"}

from IPython.display import HTML, display
import base64

if "MOVIE_PATH" not in globals():
    raise RuntimeError("Run section 11 first.")
_movie = Path(MOVIE_PATH)
if not _movie.exists():
    raise RuntimeError(f"{_movie} is gone — re-run section 11.")
print(f"{_movie}  ({_movie.stat().st_size / 2**20:.1f} MB)")

# A five minute 1080p file is far too big to embed, so preview a downscaled excerpt
# from the deepest part of the dive and leave the real file on disk.
if PREVIEW_SECONDS > 0:
    _prev = _movie.with_name("preview.mp4")
    _start = max(0.0, CFG.duration + CFG.hold_end - PREVIEW_SECONDS)
    subprocess.run([FFMPEG, "-y", "-loglevel", "error", "-ss", str(_start),
                    "-i", str(_movie), "-t", str(PREVIEW_SECONDS),
                    "-vf", "scale=640:-2", "-c:v", "libx264", "-crf", "26",
                    "-pix_fmt", "yuv420p", str(_prev)], check=False)
    if _prev.exists():
        _b64 = base64.b64encode(_prev.read_bytes()).decode()
        display(HTML(f'<video width="640" controls loop autoplay muted '
                     f'src="data:video/mp4;base64,{_b64}"></video>'
                     f'<p style="font-family:monospace;font-size:12px">last '
                     f'{PREVIEW_SECONDS:g}s, downscaled — the full-resolution file is '
                     f'at {_movie}</p>'))

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _dest = Path(DRIVE_FOLDER); _dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(_movie, _dest / _movie.name)
    print(f"copied to {_dest / _movie.name}")

if DOWNLOAD:
    from google.colab import files
    files.download(str(_movie))

#@markdown ---
#@markdown **Uploading to YouTube.** The file is already H.264 / yuv420p with
#@markdown `+faststart`, which is what YouTube wants. Keep `CRF` at 16 or lower for a
#@markdown deep zoom: the constantly moving fine detail is the hardest thing there is
#@markdown for a video codec, and YouTube re-encodes whatever you send, so upload with
#@markdown more quality than you need. 1440p or 4K also gets you YouTube's better
#@markdown encoder ladder even if the detail is 1080p-ish.